# GISLR — BiLSTM: exact replica vs curated-feature variant

**What this notebook does.** Two bidirectional-LSTM arms, trained and
evaluated identically, on the current canonical split:

- **`bilstm_base`** — the production `sb.recognize.architectures.BiLSTM`
  class, **unmodified**, fed `sb.recognize.features.base_v1`'s ME-126/xy raw
  landmarks (252-dim) — the exact architecture and feature pipeline every
  historical BiLSTM registry run used. Every one of those runs predates the
  2026-09-16 canonical-split reset (`registry/index.csv`: all 7 BiLSTM
  entries have `split_n_val=9448`, the retired split), so there is currently
  no BiLSTM number on the split everything else in this repo is now measured
  against. This arm exists to produce one, without changing the model at
  all.
- **`bilstm_curated`** — `sb.recognize.interp.models.LandmarkBiLSTM` (new
  this notebook): same bidirectional-LSTM idea and the same fwd-last/
  bwd-first readout convention as the production class, but with the
  attention gate + generalized dims every other interp-track model has, fed
  `sb.recognize.interp.features_curated`'s ME-126 + xy + 28 joint angles +
  12 relational distances (922-dim) — the "hidden variables" (angles and the
  other engineered features) from this session's prior interpretability
  work (`docs/reports/feature-discriminability.md`,
  `docs/reports/curated-features.md`).

**Both arms are OFFLINE-ONLY** — bidirectional means the backward pass reads
future frames, so neither is a deployment candidate (same constraint
`sb.recognize.architectures.BiLSTM`'s own docstring states). This experiment
prices what bidirectionality, and separately what the engineered features,
buy at the accuracy ceiling — it does not produce a streaming model.

**What "exact replica" means here, precisely**: the *architecture* (unmodified
`BiLSTM` class) and *feature pipeline* (`base_v1`, ME-126, xy) are identical
to every historical registry run. The *training regime* (hyperparameters
below) matches `gislr.training.json`'s `shared` block exactly, but this
notebook trains a **single final fit** (small internal-val carve-out for
early stopping) rather than going through the production
registry-writing driver — so this run is architecture/feature-exact but is
**not** written to `registry/`. If a literal new registry entry is wanted
instead, that is simpler and separate: (re-)run
`gislr.1.models.training.ipynb`'s existing `bilstm`/`ME_126`/`xy` config,
which already resolves to the current split automatically.

**Pipeline stage vs. diagnostic.** Self-contained under `sb.recognize.interp`
(TODO §3.6), not part of `ARCHS`/the registry — same status as
`landmark-importance.ipynb`/`curated-features.ipynb`.

**Artifacts**

| Path | Content |
|---|---|
| `data/cache/gislr/features/base_v1/<key>/` | ME-126/xy raw feature cache (train + test), content-addressed |
| `data/cache/gislr/features/landmark_curated_v1/<key>/` | curated feature cache (already built by the prior experiment) |
| `data/cache/gislr/bilstm_curated_runs/<arm>/{final.pt,test_predictions.npz}` | per-arm artifacts (gitignored) |
| `data/cache/gislr/bilstm_curated_runs/topn_summary.csv` | top-1/3/5 accuracy per arm |
| `docs/reports/bilstm-curated.md` | write-up, **after** this notebook runs |


In [1]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR
from sb.core.subsets import SUBSETS
from sb.recognize.architectures import BiLSTM
from sb.recognize.features import base_v1 as BFEAT
from sb.recognize.interp import features_curated as CFEAT
from sb.recognize.interp.geometry import RELATIONAL_DIM
from sb.recognize.interp.models import LandmarkBiLSTM
from sb.recognize.interp.train import (
    load_split_arrays,
    make_fold_loader,
    make_row_tracked_loader,
    predict_probs_indexed,
    train_fold,
)
from sb.recognize.sources import get_source

CONFIG_PATH = EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.bilstm-curated.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

SEED = CONFIG["seed"]
HYP = CONFIG["shared"]  # every training hyperparameter comes from the config, never this cell
FINAL_VAL_FRACTION = CONFIG["final_fit_val_fraction"]
TOP_N = CONFIG["top_n"]
ARMS = ["bilstm_base", "bilstm_curated"]

RUN_ROOT = CACHE_DIR / "gislr" / "bilstm_curated_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
for arm in ARMS:
    (RUN_ROOT / arm / "assets").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # same as sb.recognize.train's production driver -- cuDNN
                                        # picks the fastest algorithm for these fixed input shapes
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"config: {CONFIG_PATH}")
print(f"device: {DEVICE}")
print(f"run artifacts: {RUN_ROOT}")
print(f"bilstm_base feature_dim: 252 (ME-126 x xy, base_v1)")
print(f"bilstm_curated feature_dim: {CFEAT.FEATURE_DIM} (= {CFEAT.PER_LANDMARK_DIM} "
      f"per-landmark + {CFEAT.ANGLE_DIM} angle + {RELATIONAL_DIM} relational)")
print(f"epochs cap={HYP['epochs']} - batch_size={HYP['batch_size']} - top_n={TOP_N}")


config: C:\Users\user2\repository\experiments\recognition\configs\gislr.bilstm-curated.json
device: cuda
run artifacts: C:\Users\user2\repository\data\cache\gislr\bilstm_curated_runs
bilstm_base feature_dim: 252 (ME-126 x xy, base_v1)
bilstm_curated feature_dim: 922 (= 882 per-landmark + 28 angle + 12 relational)
epochs cap=300 - batch_size=1024 - top_n=[1, 3, 5]


## 1. Feature caches (one per arm)

`bilstm_base` reuses `sb.recognize.features.base_v1` (ME-126, xy, raw) —
exactly the pipeline every historical BiLSTM registry run trained on.
`bilstm_curated` reuses `sb.recognize.interp.features_curated` — already
built and cached by `gislr.1.models.curated-features.ipynb`, so this cell
should skip straight through for it.

In [2]:
# ============================================================
# Canonical split + both feature caches (train.csv, test.csv)
# ============================================================
ds = get_source("gislr")
DATA_DIR = ds.resolve_dir()
sign2idx = ds.label_map(DATA_DIR)
idx2sign = {v: k for k, v in sign2idx.items()}
N_CLASSES = len(sign2idx)

train_split, test_split = ds.canonical_split(DATA_DIR, sign2idx)
print(f"train.csv: {len(train_split)} videos - test.csv: {len(test_split)} videos - {N_CLASSES} classes")


def _tqdm_progress(label: str):
    """One tqdm bar per (arm, split) feature-cache build; skip-if-exists
    caches call `progress` zero times, so no bar appears for those."""
    state = {"bar": None, "last": 0}

    def _progress(done, total):
        if state["bar"] is None:
            state["bar"] = tqdm(total=total, desc=f"{label} feature cache", dynamic_ncols=True)
        state["bar"].update(done - state["last"])
        state["last"] = done
        if done >= total - 1:
            state["bar"].close()

    return _progress


ME_126 = SUBSETS["ME_126"]

# bilstm_base: base_v1, ME-126, xy (252-dim) -- the historical BiLSTM feature pipeline
base_tr_data_path, base_tr_off_path = BFEAT.build_cache(
    train_split, "train", ME_126, "xy", DATA_DIR, progress=_tqdm_progress("base_v1 train")
)
base_te_data_path, base_te_off_path = BFEAT.build_cache(
    test_split, "test", ME_126, "xy", DATA_DIR, progress=_tqdm_progress("base_v1 test")
)
BASE_FEATURE_DIM = len(ME_126) * 2  # xy only
print(f"base_v1 cache dir: {BFEAT.cache_dir(ME_126, 'xy', DATA_DIR)}  (feature_dim={BASE_FEATURE_DIM})")

# bilstm_curated: features_curated, ME-126 + xy + angles + relational (922-dim)
cur_tr_data_path, cur_tr_off_path = CFEAT.build_cache(
    train_split, "train", DATA_DIR, progress=_tqdm_progress("curated train")
)
cur_te_data_path, cur_te_off_path = CFEAT.build_cache(
    test_split, "test", DATA_DIR, progress=_tqdm_progress("curated test")
)
print(f"curated cache dir: {CFEAT.cache_dir(DATA_DIR)}  (feature_dim={CFEAT.FEATURE_DIM})")


train.csv: 75581 videos - test.csv: 18896 videos - 250 classes


base_v1 test feature cache:   0%|          | 0/18896 [00:00<?, ?it/s]

test/me126-xy [4ffb3148942dca48]: cached 18896 videos, 0.73 GB (7s)
base_v1 cache dir: C:\Users\user2\repository\data\cache\gislr\features\base_v1\4ffb3148942dca48  (feature_dim=252)


curated train feature cache:   0%|          | 0/75581 [00:00<?, ?it/s]

train/landmark_curated_v1 [b8e8de2c8911b751]: cached 75581 videos, 10.56 GB (229s)


curated test feature cache:   0%|          | 0/18896 [00:00<?, ?it/s]

test/landmark_curated_v1 [b8e8de2c8911b751]: cached 18896 videos, 2.65 GB (30s)
curated cache dir: C:\Users\user2\repository\data\cache\gislr\features\landmark_curated_v1\b8e8de2c8911b751  (feature_dim=922)


## 2. Model factories

In [3]:
# ============================================================
# Model factories -- every hyperparameter from CONFIG's shared block
# ============================================================
def make_model(arm: str):
    if arm == "bilstm_base":
        return BiLSTM(
            input_size=BASE_FEATURE_DIM, hidden_size=HYP["hidden_size"],
            num_layers=HYP["num_layers"], num_classes=N_CLASSES, dropout=HYP["dropout"],
        )
    return LandmarkBiLSTM(
        proj_size=CONFIG["arms"]["bilstm_curated"]["proj_size"],
        hidden_size=HYP["hidden_size"], num_layers=HYP["num_layers"],
        num_classes=N_CLASSES, dropout=HYP["dropout"],
        n_landmarks=CFEAT.N_LANDMARKS, channels_per_landmark=CFEAT.CHANNELS_PER_LANDMARK,
        angle_dim=CFEAT.ANGLE_DIM, relational_dim=RELATIONAL_DIM,
    )


n_params = {arm: sum(p.numel() for p in make_model(arm).parameters()) for arm in ARMS}
pd.Series(n_params, name="n_params").to_frame()


,n_params
bilstm_base,2751218
bilstm_curated,2997676


## 3. Final fit (one model per arm, trained on (nearly) all of `train.csv`)

Single final fit, no k-fold — same convention `curated-features.ipynb` §3
used. Internal-val carve-out drives early stopping/LR only.

In [4]:
# ============================================================
# Final fit: one model per arm
# ============================================================
ARM_FEATURE_DIM = {"bilstm_base": BASE_FEATURE_DIM, "bilstm_curated": CFEAT.FEATURE_DIM}
ARM_TRAIN_CACHE = {"bilstm_base": (base_tr_data_path, base_tr_off_path),
                    "bilstm_curated": (cur_tr_data_path, cur_tr_off_path)}
ARM_TEST_CACHE = {"bilstm_base": (base_te_data_path, base_te_off_path),
                   "bilstm_curated": (cur_te_data_path, cur_te_off_path)}


def hyp_for_arm(arm: str) -> dict:
    """`shared` + this arm's `overrides` (if any) -- bilstm_base has none, so
    it trains under gislr.training.json's exact regime; bilstm_curated
    overrides batch_size/lr to use more of the GPU (no historical regime to
    stay faithful to, see the config's `notes`)."""
    return {**HYP, **CONFIG["arms"][arm].get("overrides", {})}


tr_idx_final, val_idx_final = train_test_split(
    np.arange(len(train_split)), test_size=FINAL_VAL_FRACTION,
    stratify=train_split["label"], random_state=SEED,
)
print(f"final fit: {len(tr_idx_final)} train / {len(val_idx_final)} internal-val "
      f"(early-stopping signal only -- never a reported metric)")

final_models = {}
arm_train_arrays = {}
for arm in ARMS:
    arm_hyp = hyp_for_arm(arm)
    if arm_hyp != HYP:
        print(f"{arm}: overrides -> { {k: v for k, v in arm_hyp.items() if HYP.get(k) != v} }")

    data_path, off_path = ARM_TRAIN_CACHE[arm]
    train_data, train_off, train_labels = load_split_arrays(data_path, off_path, train_split)
    arm_train_arrays[arm] = (train_data, train_off, train_labels)

    ckpt = RUN_ROOT / arm / "final.pt"
    tr_loader = make_fold_loader(train_data, train_off, train_labels, tr_idx_final,
                                  arm_hyp["batch_size"], True, SEED, feature_dim=ARM_FEATURE_DIM[arm])
    va_loader = make_fold_loader(train_data, train_off, train_labels, val_idx_final,
                                  arm_hyp["batch_size"], False, SEED, feature_dim=ARM_FEATURE_DIM[arm])
    bar = tqdm(total=arm_hyp["epochs"], desc=f"{arm} - final fit", dynamic_ncols=True)
    model, history, best_val_acc, best_state = train_fold(
        arch=arm, model_fn=lambda a=arm: make_model(a), train_loader=tr_loader, val_loader=va_loader,
        hyp=arm_hyp, device=DEVICE, ckpt_path=ckpt, bar=bar, bar_prefix="final",
    )
    bar.close()
    model.load_state_dict(best_state)
    model.eval()
    final_models[arm] = model
    (RUN_ROOT / arm / "history.json").write_text(json.dumps(history))
    print(f"{arm}: final fit best internal-val acc {best_val_acc:.4f}")

final fit: 71801 train / 3780 internal-val (early-stopping signal only -- never a reported metric)


bilstm_base - final fit:   0%|          | 0/300 [00:00<?, ?it/s]

bilstm_base: final fit best internal-val acc 0.7360
bilstm_curated: overrides -> {'batch_size': 4096, 'lr': 0.005656}


bilstm_curated - final fit:   0%|          | 0/300 [00:00<?, ?it/s]

bilstm_curated: final fit best internal-val acc 0.0042


## 4. Held-out test-set evaluation + top-N accuracy

One pass per arm over the untouched canonical `test.csv`, same top-N
protocol as `curated-features.ipynb` §4.

In [5]:
# ============================================================
# ONE evaluation pass per final model, on the untouched canonical test.csv
# ============================================================
arm_results = {}
topn_rows = []
for arm in ARMS:
    data_path, off_path = ARM_TEST_CACHE[arm]
    test_data, test_off, test_labels_arr = load_split_arrays(data_path, off_path, test_split)

    loader = make_row_tracked_loader(
        test_data, test_off, test_labels_arr, np.arange(len(test_split)), hyp_for_arm(arm)["batch_size"],
        feature_dim=ARM_FEATURE_DIM[arm],
    )
    rows, probs, ret_labels = predict_probs_indexed(
        final_models[arm], "bilstm", loader, DEVICE, desc=f"{arm} - test.csv eval"
    )
    order = np.argsort(rows)
    labels_ordered, probs_ordered = ret_labels[order], probs[order]

    ranked = np.argsort(-probs_ordered, axis=-1)
    for n in TOP_N:
        hit = (ranked[:, :n] == labels_ordered[:, None]).any(axis=1)
        topn_rows.append({"arm": arm, "top_n": n, "accuracy": float(hit.mean())})

    arm_results[arm] = dict(labels=labels_ordered, probs=probs_ordered, ranked=ranked,
                             test_off=test_off, test_data=test_data)
    np.savez(RUN_ROOT / arm / "test_predictions.npz",
             labels=labels_ordered, probs=probs_ordered, ranked_top5=ranked[:, :5])

topn_df = pd.DataFrame(topn_rows).pivot(index="arm", columns="top_n", values="accuracy")
topn_df.columns = [f"top_{n}_acc" for n in topn_df.columns]
topn_df.to_csv(RUN_ROOT / "topn_summary.csv")
topn_df


bilstm_base - test.csv eval:   0%|          | 0/19 [00:00<?, ?it/s]

bilstm_curated - test.csv eval:   0%|          | 0/5 [00:00<?, ?it/s]

,top_1_acc,top_3_acc,top_5_acc
arm,,,
bilstm_base,0.737140,0.861452,0.893575
bilstm_curated,0.004181,0.012172,0.020481


## 5. Confusable-pair check

Same 16 documented pairs as `curated-features.ipynb` §5 /
`docs/reports/pair-similarity.md`.

In [6]:
# ============================================================
# Confusable-pair rank-recovery, per arm
# ============================================================
CONFUSABLE_PAIRS = [
    ("awake", "wake"), ("lips", "mouth"), ("hear", "listen"), ("pen", "pencil"),
    ("gift", "give"), ("cut", "scissors"), ("finger", "wait"), ("stay", "that"),
    ("duck", "goose"), ("animal", "have"), ("cat", "kitty"), ("sleep", "sleepy"),
    ("nap", "sleep"), ("bed", "bedroom"), ("dry", "dryer"), ("look", "see"),
]

for arm in ARMS:
    labels_ordered = arm_results[arm]["labels"]
    ranked = arm_results[arm]["ranked"]
    print(f"\n=== {arm} ===")
    rows = []
    for a, b in CONFUSABLE_PAIRS:
        if a not in sign2idx or b not in sign2idx:
            continue
        for true_sign, other_sign in ((a, b), (b, a)):
            true_idx, other_idx = sign2idx[true_sign], sign2idx[other_sign]
            mask = labels_ordered == true_idx
            if mask.sum() == 0:
                continue
            r = ranked[mask]
            top1 = (r[:, 0] == true_idx).mean()
            top1_is_other = (r[:, 0] == other_idx).mean()
            top2_recovers = ((r[:, 0] == other_idx) & (r[:, 1] == true_idx)).sum()
            rows.append({
                "true_sign": true_sign, "confused_with": other_sign, "n": int(mask.sum()),
                "top1_acc": top1, "rank1_is_other": top1_is_other,
                "recovered_at_rank2": int(top2_recovers),
            })
    print(pd.DataFrame(rows).to_string(index=False))



=== bilstm_base ===
true_sign confused_with  n  top1_acc  rank1_is_other  recovered_at_rank2
    awake          wake 80  0.487500        0.400000                  28
     wake         awake 80  0.462500        0.412500                  31
     lips         mouth 81  0.567901        0.271605                  21
    mouth          lips 78  0.512821        0.333333                  20
     hear        listen 81  0.629630        0.148148                  11
   listen          hear 83  0.698795        0.204819                  16
      pen        pencil 80  0.600000        0.237500                  13
   pencil           pen 78  0.576923        0.243590                  14
     gift          give 79  0.607595        0.101266                   4
     give          gift 69  0.391304        0.231884                  10
      cut      scissors 74  0.648649        0.175676                  10
 scissors           cut 74  0.689189        0.162162                   8
   finger          wait 76  0.

## 6. Comparison against everything else on the current split

Both arms, this session's curated-feature DNN/LSTM
(`docs/reports/curated-features.md`), and every canonical registry run on
the current split (`registry/index.csv`, queried live so this table can't
go stale).

In [7]:
# ============================================================
# Situate this notebook's numbers against the rest of the current-split leaderboard
# ============================================================
from sb.mlops.query import query_runs

canonical_current_split = query_runs(
    where="dataset = 'gislr' AND split.n_val = 18896", order_by="accuracy DESC"
)
print("Canonical registry runs on the current split:")
print(canonical_current_split[["architecture", "subset", "coords", "accuracy"]].to_string(index=False)
      if len(canonical_current_split) else "(none)")

print()
print("This notebook (bilstm_base / bilstm_curated), top-1:")
print(topn_df["top_1_acc"])
print()
print("For reference (docs/reports/curated-features.md, same held-out test.csv):")
print(pd.DataFrame({
    "arch": ["dnn (full-543, landmark_interp_v1)", "lstm (full-543, landmark_interp_v1)",
             "dnn (curated, landmark_curated_v1)", "lstm (curated, landmark_curated_v1)"],
    "top_1_acc": [0.7102, 0.7055, 0.5776, 0.6909],
}).to_string(index=False))


Canonical registry runs on the current split:
architecture subset coords  accuracy
         gru ME_132     xy  0.751746
         gru ME_126     xy  0.745025
         gru FP_118     xy  0.742485

This notebook (bilstm_base / bilstm_curated), top-1:
arm
bilstm_base       0.737140
bilstm_curated    0.004181
Name: top_1_acc, dtype: float64

For reference (docs/reports/curated-features.md, same held-out test.csv):
                               arch  top_1_acc
 dnn (full-543, landmark_interp_v1)     0.7102
lstm (full-543, landmark_interp_v1)     0.7055
 dnn (curated, landmark_curated_v1)     0.5776
lstm (curated, landmark_curated_v1)     0.6909


## 7. Verdict

Fill in after running: does the curated feature set help `LandmarkBiLSTM`
the way it helped the causal LSTM in `curated-features.ipynb` (−1.46pp vs
full-543) or hurt it the way it hurt the DNN (−13.26pp)? Does bidirectionality
alone (comparing `bilstm_base` to the current-split canonical GRU runs, both
on raw ME-126/xy) buy anything over the causal deployment architecture, now
that both are finally measured on the same split?

In [8]:
# ============================================================
# Verdict table (fill in narrative above after running)
# ============================================================
print(topn_df)
print()
print("Params:", n_params)
print()
print("Artifacts:")
for arm in ARMS:
    print(f"  {RUN_ROOT / arm / 'final.pt'}")
    print(f"  {RUN_ROOT / arm / 'test_predictions.npz'}")
print(f"  {RUN_ROOT / 'topn_summary.csv'}")


                top_1_acc  top_3_acc  top_5_acc
arm                                            
bilstm_base      0.737140   0.861452   0.893575
bilstm_curated   0.004181   0.012172   0.020481

Params: {'bilstm_base': 2751218, 'bilstm_curated': 2997676}

Artifacts:
  C:\Users\user2\repository\data\cache\gislr\bilstm_curated_runs\bilstm_base\final.pt
  C:\Users\user2\repository\data\cache\gislr\bilstm_curated_runs\bilstm_base\test_predictions.npz
  C:\Users\user2\repository\data\cache\gislr\bilstm_curated_runs\bilstm_curated\final.pt
  C:\Users\user2\repository\data\cache\gislr\bilstm_curated_runs\bilstm_curated\test_predictions.npz
  C:\Users\user2\repository\data\cache\gislr\bilstm_curated_runs\topn_summary.csv
